In [1]:
from pyspark import SparkConf, SparkContext
conf = SparkConf().setAppName("Prova_esame")
sc = SparkContext(conf=conf)

# Task 1

In [3]:
outputPath1="./output1/"
videoLecturesRDD=sc.textFile("./data/VideoLectures.txt")
onlineCoursesRDD=sc.textFile("./data/OnlineCourses.txt")

In [21]:
cleanedLecturesRDD=videoLecturesRDD.map(lambda x: (x.split(",")[3],int(x.split(",")[2]))) \
    .reduceByKey(lambda a,b: a+b).mapValues(lambda x: 1 if x>600 else 0)

cleanedCoursesRDD=onlineCoursesRDD.map(lambda x: (x.split(",")[0],x.split(",")[2]))

joinedRDD=cleanedLecturesRDD.join(cleanedCoursesRDD).map(lambda x: (x[1][1],(x[1][0],1))) \
    .reduceByKey(lambda a, b: (a[0]+b[0],a[1]+b[1])).filter(lambda x: x[1][0]/x[1][1]>0.8) \
    .keys()

joinedRDD.saveAsTextFile(outputPath1)

#Task 2

In [42]:
outputPath2="./output2/"
usersWatchedRDD=sc.textFile("./data/UsersWatchedLectures.txt")
studentsRDD=sc.textFile("./data/Students.txt")

In [53]:
def mappaggio(row):
  userId, year= row[0], row[1][0]
  year21=0
  year22=0
  year23=0
  if year=='2021':
    year21+=1
  elif year=='2022':
    year22+=1
  else:
    year23+=1
  return (userId,(year21,year22,year23))

cleanedWatchedRDD=usersWatchedRDD.map(lambda x: (x.split(",")[0],(x.split(",")[1].split("/")[0],x.split(",")[2]))) \
    .filter(lambda x: x[1][0]=='2021' or x[1][0]=='2022' or x[1][0]=='2023') \
    .distinct().map(mappaggio)

cleanedStudentsRDD=studentsRDD.map(lambda x: (x.split(",")[0],(0,0,0)))

joined2RDD=cleanedWatchedRDD.union(cleanedStudentsRDD).reduceByKey(lambda a,b: (a[0]+b[0],a[1]+b[1],a[2]+b[2])) \
    .filter(lambda x: x[1][2]==0)
